## **Tareas a resolver:**

**Clasificación binaria: mentira o verdad**

**Predicción del hablante: de qué país es el mensaje**

Antes de empezar a resolver las tareas vamos a intentar arreglar el problema del desbalance de clases

**1. Carga y análisis inicial del dataset**
- Cargar datos
- Expandir mensajes
- Analizar distribución de clases
- Identificar desbalance

In [1]:
import pandas as pd
import json

data = pd.read_parquet('data/train_preprocessed.parquet')
df_expanded = data.explode(["messages", "sender_labels", "receiver_labels"])
df_expanded.head()

,messages,sender_labels,receiver_labels,speakers,receivers,absolute_message_index,relative_message_index,seasons,years,game_score,game_score_delta,players,game_id,text_clean,tokens,lemmas
0,Germany!\n\nJust the person I want to speak wi...,True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,germany just the person i want to speak with i...,"['germany', 'person', 'want', 'speak', 'somewh...","['germany', 'person', 'want', 'speak', 'somewh..."
1,"You've whet my appetite, Italy. What's the sug...",True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,youve whet my appetite italy whats the suggestion,"['ve', 'whet', 'appetite', 'italy', 's', 'sugg...","['ve', 'whet', 'appetite', 'italy', 's', 'sugg..."
2,It seems like there are a lot of ways that cou...,True,True,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,it seems like there are a lot of ways that cou...,"['like', 'lot', 'ways', 'wrong', 'nt', 'france...","['like', 'lot', 'way', 'wrong', 'not', 'france..."
3,"Yeah, I can’t say I’ve tried it and it works, ...",True,NOANNOTATION,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,yeah i can t say i ve tried it and it works ca...,"['yeah', 't', 've', 'tried', 'works', 'cause',...","['yeah', 't', 've', 'try', 'work', 'cause', 'v..."
4,I am just sensing that you don’t like this ide...,True,NOANNOTATION,"['italy', 'germany', 'italy', 'germany', 'ital...","['germany', 'italy', 'germany', 'italy', 'germ...","[74, 76, 86, 87, 89, 92, 97, 117, 119, 121, 12...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Spring', 'Spring', 'Spring', 'Spring', 'Spri...","['1901', '1901', '1901', '1901', '1901', '1901...","['3', '3', '3', '3', '3', '3', '3', '3', '3', ...","['0', '0', '0', '0', '0', '0', '0', '0', '0', ...","['italy', 'germany']",1,i am just sensing that you don t like this ide...,"['sensing', 'don', 't', 'like', 'idea', 'shall...","['sense', 'don', 't', 'like', 'idea', 'shall',..."


**2. Análisis del desbalance de clases**
- Distribución de `sender_labels`
- Distribución de `receiver_labels`

In [2]:
print("Distribución sender_labels:")
print(df_expanded["sender_labels"].value_counts())
print("\nDistribución receiver_labels:")
print(df_expanded["receiver_labels"].value_counts())

Distribución sender_labels:
sender_labels
True     11372
False      522
Name: count, dtype: int64

Distribución receiver_labels:
receiver_labels
True            10390
NOANNOTATION      989
False             515
Name: count, dtype: int64


**3. Corrección del desbalance de clases**
- Uso de *class weights*
- Preparación para entrenamiento

In [3]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# -------- sender_labels (binario) --------
y_sender = df_expanded["sender_labels"]

sender_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_sender),
    y=y_sender
)

print("\nClass weights para sender_labels:")
print(sender_class_weights)


# -------- receiver_labels (multiclase) --------
y_receiver = df_expanded["receiver_labels"]

receiver_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_receiver),
    y=y_receiver
)

print("\nClass weights para receiver_labels:")
print(receiver_class_weights)


Class weights para sender_labels:
[11.39272031  0.52295111]

Class weights para receiver_labels:
[7.69838188 4.00876306 0.38158486]


**Resultados**

Durante el entrenamiento de los modelos, utilizaremos estos pesos para que la función de pérdida multiplique el error de cada ejemplo por el peso correspondiente a su clase. De esta manera, el optimizador ajustará los parámetros del modelo para minimizar la pérdida ponderada, lo que mejora significativamente la capacidad del modelo para predecir correctamente las clases minoritarias.

In [15]:
# ======================================
# Preparación de vectores Word2Vec por mensaje
# ======================================

import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from tqdm import tqdm
import ast

tqdm.pandas()

# ------------------------------
# Cargar DataFrame y Word2Vec
# ------------------------------
df = pd.read_parquet("data/train_preprocessed.parquet")
w2v = Word2Vec.load("diplomacy/models/embeddings/word2vec.model")

# ------------------------------
# Convertir tokens de string a lista (si es necesario)
# ------------------------------
if isinstance(df["tokens"].iloc[0], str):
    df["tokens"] = df["tokens"].apply(ast.literal_eval)

# ------------------------------
# Comprobación de tokens OOV
# ------------------------------
all_tokens = df["tokens"].explode()
in_vocab = all_tokens.apply(lambda w: w in w2v.wv)

print("Total de tokens:", len(all_tokens))
print("Tokens con embedding:", in_vocab.sum(), f"({in_vocab.mean()*100:.2f}%)")
print("Tokens fuera del vocabulario:", len(all_tokens) - in_vocab.sum())

# ------------------------------
# Función para generar vector promedio por mensaje
# ------------------------------
def message_to_vector(tokens, model):
    """
    Convierte una lista de tokens en un vector promedio usando Word2Vec.
    Tokens fuera del vocabulario se ignoran.
    """
    vectors = [model.wv[t] for t in tokens if t in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# ------------------------------
# Generar vectores para todos los mensajes
# ------------------------------
df["w2v_vector"] = df["tokens"].progress_apply(lambda tokens: message_to_vector(tokens, w2v))

# ------------------------------
# Verificar resultado
# ------------------------------
print("Primer vector (primeros 5 valores):", df["w2v_vector"].iloc[0][:5])
print("Shape de un vector:", df["w2v_vector"].iloc[0].shape)


Total de tokens: 98511
Tokens con embedding: 90643 (92.01%)
Tokens fuera del vocabulario: 7868


100%|██████████████████████████████████████████████████████| 11894/11894 [00:00<00:00, 36967.31it/s]

Primer vector (primeros 5 valores): [ 0.02355975 -0.10959967 -0.09957806  0.17633413  0.06859857]
Shape de un vector: (200,)


## **Shallow ML**
En esta sección vamos a intentar resolver las tareas con técnicas de shallow ML

### **1. Clasificación binaria**
   

In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from scipy.sparse import load_npz
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
import joblib

# --------------------------
# 1. Cargar datos
# --------------------------
X_tfidf = load_npz("diplomacy/models/representations/X_tfidf_train.npz")
X_bow   = load_npz("diplomacy/models/representations/X_bow_train.npz")

df = pd.read_parquet("data/train_preprocessed.parquet")

# Convertir etiquetas sender_labels a binario 0/1
y = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0})

print("Distribución de clases:")
print(y.value_counts())

# --------------------------
# 2. Pesos de clase
# --------------------------
class_weight_sender = {
    0: 11.39272031,   # False = minoritaria
    1: 0.52295111     # True = mayoritaria
}

# Para XGBoost usamos scale_pos_weight como recomendación oficial
pos_weight = class_weight_sender[0] / class_weight_sender[1]
print("\nscale_pos_weight =", pos_weight)

# --------------------------
# 3. División train/val
# --------------------------
X_train_tfidf, X_val_tfidf, y_train, y_val = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

X_train_bow, X_val_bow, _, _ = train_test_split(
    X_bow, y, test_size=0.2, random_state=42, stratify=y
)

# ==================================================
# 4. LOGISTIC REGRESSION (TF-IDF)
# ==================================================
lr = LogisticRegression(
    max_iter=3000,
    class_weight=class_weight_sender
)

lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_val_tfidf)

print("\n=== Logistic Regression (TF-IDF) ===")
print(classification_report(y_val, y_pred_lr))

joblib.dump(lr, "diplomacy/models/shallow/logreg_tfidf_weighted.joblib")

# ==================================================
# 5. LINEAR SVM (BoW)
# ==================================================
svm = LinearSVC(
    class_weight=class_weight_sender,
    max_iter=3000
)

svm.fit(X_train_bow, y_train)
y_pred_svm = svm.predict(X_val_bow)

print("\n=== Linear SVM (BoW) ===")
print(classification_report(y_val, y_pred_svm))

joblib.dump(svm, "diplomacy/models/shallow/svm_bow_weighted.joblib")

# ==================================================
# 6. XGBOOST (TF-IDF) – con oversampling para evitar collapse
# ==================================================
print("\nAplicando oversampling SOLO para XGBoost...")

ros = RandomOverSampler(random_state=42)
X_train_tfidf_bal, y_train_bal = ros.fit_resample(X_train_tfidf, y_train)

print("Distribución balanceada para XGBoost:")
print(pd.Series(y_train_bal).value_counts())

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=pos_weight,
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train_tfidf_bal, y_train_bal)
y_pred_xgb = xgb.predict(X_val_tfidf)

print("\n=== XGBoost (TF-IDF + Oversampling + scale_pos_weight) ===")
print(classification_report(y_val, y_pred_xgb))

joblib.dump(xgb, "diplomacy/models/shallow/xgb_tfidf_weighted.joblib")

Distribución de clases:
sender_labels
1    11372
0      522
Name: count, dtype: int64

scale_pos_weight = 21.78544053573191

=== Logistic Regression (TF-IDF) ===
              precision    recall  f1-score   support

           0       0.11      0.22      0.15       104
           1       0.96      0.92      0.94      2275

    accuracy                           0.89      2379
   macro avg       0.54      0.57      0.55      2379
weighted avg       0.93      0.89      0.91      2379


=== Linear SVM (BoW) ===
              precision    recall  f1-score   support

           0       0.07      0.09      0.08       104
           1       0.96      0.95      0.95      2275

    accuracy                           0.91      2379
   macro avg       0.52      0.52      0.52      2379
weighted avg       0.92      0.91      0.92      2379


Aplicando oversampling SOLO para XGBoost...
Distribución balanceada para XGBoost:
sender_labels
1    9097
0    9097
Name: count, dtype: int64

=== XGBoost (T

['diplomacy/models/shallow/xgb_tfidf_weighted.joblib']

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV
from scipy.sparse import load_npz, hstack
from imblearn.over_sampling import SMOTE
from gensim.models import Word2Vec
from sklearn.preprocessing import StandardScaler
import joblib

# ======================================
# 1. CARGA DE DATOS Y ETIQUETAS
# ======================================
print("Cargando datos...")
X_tfidf = load_npz("diplomacy/models/representations/X_tfidf_train.npz")
df = pd.read_parquet("data/train_preprocessed.parquet")

# Etiquetas en formato binario
y = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0})

# Tus class weights calculados previamente
sender_class_weights = {
    0: 11.39272031,
    1: 0.52295111
}

# Cargar Word2Vec
w2v = Word2Vec.load("diplomacy/models/embeddings/word2vec.model")

# ======================================
# 2. GENERAR WORD2VEC PROMEDIADO POR TEXTO
# ======================================
def text_to_w2v(tokens):
    tokens = eval(tokens) if isinstance(tokens, str) else tokens
    vecs = [w2v.wv[word] for word in tokens if word in w2v.wv]
    if len(vecs) == 0:
        return np.zeros(w2v.vector_size)
    return np.mean(vecs, axis=0)

print("Generando embeddings Word2Vec...")
w2v_features = np.vstack(df["tokens"].apply(text_to_w2v).values)

# Escalado para combinar con TF-IDF
scaler = StandardScaler()
w2v_features_scaled = scaler.fit_transform(w2v_features)

# Convertir a sparse y concatenar con TF-IDF
w2v_sparse = np.nan_to_num(w2v_features_scaled)
X_combined = hstack([X_tfidf, w2v_sparse])

# ======================================
# 3. TRAIN/VAL SPLIT
# ======================================
X_train, X_val, y_train, y_val = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

# ======================================
# 4. SMOTE (mejor que RandomOverSampler)
# ======================================
print("Aplicando SMOTE...")
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Distribución tras SMOTE:")
print(pd.Series(y_train_smote).value_counts())

# ======================================
# 5. GRIDSEARCH PARA LOGISTIC REGRESSION
# ======================================
print("\nBuscando mejores hiperparámetros (LogReg)...")

params = {
    "C": [0.1, 1, 5],
    "penalty": ["l2"],
    "solver": ["liblinear", "lbfgs"],
    "class_weight": [
        None,
        "balanced",
        sender_class_weights
    ]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=3000),
    param_grid=params,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1
)

grid.fit(X_train_smote, y_train_smote)

print("Mejores parámetros:", grid.best_params_)

best_lr = grid.best_estimator_
y_pred_lr = best_lr.predict(X_val)

print("\nResultados Logistic Regression + SMOTE + Word2Vec + GridSearch:")
print(classification_report(y_val, y_pred_lr))

joblib.dump(best_lr, "diplomacy/models/shallow/logreg_advanced.joblib")

# ======================================
# 6. BUSCAR UMBRAL ÓPTIMO PARA CLASE 0
# ======================================
print("\nBuscando umbral óptimo para clase minoritaria...")

y_probs = best_lr.predict_proba(X_val)[:, 1]

best_thr, best_f1 = 0, 0
for thr in np.arange(0.1, 0.9, 0.05):
    pred = (y_probs >= thr).astype(int)
    f1 = f1_score(y_val, pred, pos_label=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, thr

print(f"Mejor umbral = {best_thr:.2f} con F1(minoría) = {best_f1:.3f}")

final_pred = (y_probs >= best_thr).astype(int)

print("\nResultados Logistic Regression con umbral optimizado:")
print(classification_report(y_val, final_pred))

Cargando datos...
Generando embeddings Word2Vec...
Aplicando SMOTE...
Distribución tras SMOTE:
sender_labels
1    9097
0    9097
Name: count, dtype: int64

Buscando mejores hiperparámetros (LogReg)...
Mejores parámetros: {'C': 5, 'class_weight': None, 'penalty': 'l2', 'solver': 'liblinear'}

Resultados Logistic Regression + SMOTE + Word2Vec + GridSearch:
              precision    recall  f1-score   support

           0       0.15      0.16      0.16       104
           1       0.96      0.96      0.96      2275

    accuracy                           0.92      2379
   macro avg       0.55      0.56      0.56      2379
weighted avg       0.93      0.92      0.92      2379


Buscando umbral óptimo para clase minoritaria...
Mejor umbral = 0.45 con F1(minoría) = 0.158

Resultados Logistic Regression con umbral optimizado:
              precision    recall  f1-score   support

           0       0.16      0.15      0.16       104
           1       0.96      0.96      0.96      2275

   

### **2. Predicción del hablante**


In [9]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

# --------------------------
# 1. Cargar dataset preprocesado
# --------------------------
df = pd.read_parquet("data/train_preprocessed.parquet")

# --------------------------
# 2. Separar distintos speakers
# --------------------------
def flatten_speaker(x):
    if isinstance(x, list):
        return str(x[0])
    elif isinstance(x, str) and x.startswith('[') and x.endswith(']'):
        return x.strip('[]').replace("'", "").split(',')[0].strip()
    else:
        return str(x)

df['speakers'] = df['speakers'].apply(flatten_speaker)
y_speaker = df['speakers']
print("Clases finales:", y_speaker.nunique())

# --------------------------
# 4. Cargar features TF-IDF (filtrando igual)
# --------------------------
X_full = sp.load_npz("diplomacy/models/representations/X_tfidf_train.npz")
X_tfidf = X_full[df.index]

# --------------------------
# 5. Dividir train/test con estratificación
# --------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y_speaker,
    test_size=0.2,
    random_state=42,
    stratify=y_speaker
)

# --------------------------
# 6. Entrenar modelo con pesos balanceados
# --------------------------
clf = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced"
)
clf.fit(X_train, y_train)

# --------------------------
# 7. Predicciones
# --------------------------
y_pred = clf.predict(X_test)

# --------------------------
# 8. Evaluación
# --------------------------
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, digits=4))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

# ROC AUC macro OvR
y_test_bin = pd.get_dummies(y_test)
y_pred_proba = clf.predict_proba(X_test)
roc_auc = roc_auc_score(y_test_bin, y_pred_proba, average="macro", multi_class="ovr")
print(f"\nROC AUC (macro, OvR): {roc_auc:.4f}")

# --------------------------
# 9. Guardar modelo
# --------------------------
model_path = "diplomacy/models/shallow/logreg_multinomial_speaker.joblib"
os.makedirs(os.path.dirname(model_path), exist_ok=True)
joblib.dump(clf, model_path)
print(f"\nModelo guardado en: {model_path}")


Clases finales: 7

=== Classification Report ===
              precision    recall  f1-score   support

     austria     0.3103    0.3046    0.3075       325
     england     0.5077    0.3909    0.4417       591
      france     0.2051    0.3137    0.2481       204
     germany     0.2614    0.3019    0.2802       265
       italy     0.4915    0.3777    0.4271       609
      russia     0.2085    0.2658    0.2337       222
      turkey     0.1610    0.2331    0.1905       163

    accuracy                         0.3367      2379
   macro avg     0.3065    0.3125    0.3041      2379
weighted avg     0.3715    0.3367    0.3484      2379


=== Confusion Matrix ===
[[ 99  34  28  38  47  41  38]
 [ 50 231  82  66  62  51  49]
 [ 18  41  64  14  25  23  19]
 [ 32  34  28  80  54  20  17]
 [ 75  62  70  65 230  64  43]
 [ 30  34  22  19  26  59  32]
 [ 15  19  18  24  24  25  38]]

ROC AUC (macro, OvR): 0.7004

Modelo guardado en: diplomacy/models/shallow/logreg_multinomial_speaker.joblib


## **CNNs o Redes Recurrentes**

En esta sección vamos a entrenar modelos CNN para resolver las tareas

### **1. Clasificación binaria**
   

In [5]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from gensim.models import Word2Vec
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
import joblib

# --------------------------
# Paths y Config
# --------------------------
DATA_PATH = "data/train_preprocessed.parquet"
EMB_DIR = "diplomacy/models/embeddings"
OUT_DIR = "diplomacy/models/deep"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 64
EPOCHS = 8
MAX_LEN = 120
EMB_DIM = 200  # Debe coincidir con vector_size de Word2Vec

# --------------------------
# Cargar dataframe y etiquetas
# --------------------------
df = pd.read_parquet(DATA_PATH)

# Filtrar etiquetas válidas y convertir a binario
df = df[df["sender_labels"].isin(["true","false","True","False"])]
y_raw = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0})

le = LabelEncoder()
y = le.fit_transform(y_raw)

# --------------------------
# Pesos de clase
# --------------------------
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y),
    y=y
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print("Class weights para sender_labels:", class_weights.cpu().numpy())

# --------------------------
# Word2Vec
# --------------------------
w2v = Word2Vec.load(os.path.join(EMB_DIR, "word2vec.model"))

# Crear vocabulario: token -> index (0 padding, 1 OOV)
token2idx = {"<pad>":0, "<oov>":1}
for tok in w2v.wv.key_to_index.keys():
    token2idx[tok] = len(token2idx)

vocab_size = len(token2idx)

def to_indices(tokens):
    if isinstance(tokens, str):
        tokens = eval(tokens)
    idxs = [token2idx.get(tok, token2idx["<oov>"]) for tok in tokens]
    if len(idxs) >= MAX_LEN:
        return idxs[:MAX_LEN]
    else:
        return idxs + [token2idx["<pad>"]] * (MAX_LEN - len(idxs))

X_seq = np.vstack(df["tokens"].apply(lambda t: np.array(to_indices(t))).values)

# Split train/val
X_train, X_val, y_train, y_val = train_test_split(
    X_seq, y, test_size=0.2, random_state=42, stratify=y
)

# --------------------------
# Embedding matrix
# --------------------------
emb_matrix = np.random.normal(scale=0.6, size=(vocab_size, EMB_DIM)).astype(np.float32)
for tok, idx in token2idx.items():
    if tok in w2v.wv:
        emb_matrix[idx] = w2v.wv[tok]

# --------------------------
# Dataset y Dataloader
# --------------------------
class DiplomacyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = DiplomacyDataset(X_train, y_train)
val_ds = DiplomacyDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# --------------------------
# Modelo CNN
# --------------------------
class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, num_classes=2, freeze_emb=True, n_filters=100, kernel_sizes=[3,4,5]):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(emb_matrix))
        self.embedding.weight.requires_grad = not freeze_emb
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, n_filters, k) for k in kernel_sizes])
        self.fc = nn.Linear(n_filters * len(kernel_sizes), num_classes)
    def forward(self, x):
        e = self.embedding(x).permute(0,2,1)  # (B, emb_dim, L)
        convs = [torch.relu(conv(e)) for conv in self.convs]
        pools = [torch.max(c, dim=2)[0] for c in convs]
        cat = torch.cat(pools, dim=1)
        return self.fc(cat)

# --------------------------
# Entrenamiento
# --------------------------
def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_val_f1 = 0
    for epoch in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            opt.step()
        # Evaluación
        model.eval()
        y_trues, y_preds = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                logits = model(Xb)
                preds = logits.argmax(dim=1).cpu().numpy()
                y_preds.extend(preds)
                y_trues.extend(yb.cpu().numpy())
        val_f1 = f1_score(y_trues, y_preds, average="macro")
        print(f"Epoch {epoch+1}/{epochs} - val_f1_macro: {val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = model.state_dict()
    model.load_state_dict(best_state)
    return model

# --------------------------
# Entrenamiento final
# --------------------------
cnn = CNNClassifier(vocab_size=vocab_size, emb_dim=EMB_DIM, num_classes=2, freeze_emb=True)
cnn = train_model(cnn, train_loader, val_loader)
torch.save(cnn.state_dict(), os.path.join(OUT_DIR, "cnn_sender_labels_freeze.pt"))

cnn_ft = CNNClassifier(vocab_size=vocab_size, emb_dim=EMB_DIM, num_classes=2, freeze_emb=False)
cnn_ft = train_model(cnn_ft, train_loader, val_loader)
torch.save(cnn_ft.state_dict(), os.path.join(OUT_DIR, "cnn_sender_labels_finetune.pt"))

# --------------------------
# Evaluación
# --------------------------
def evaluate_model(model):
    model.eval()
    y_preds, y_trues = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            y_preds.extend(preds)
            y_trues.extend(yb.cpu().numpy())
    target_names = [str(c) for c in le.classes_]
    print(classification_report(y_trues, y_preds, target_names=target_names))
    return accuracy_score(y_trues, y_preds), f1_score(y_trues, y_preds, average="macro")

print("Evaluación CNN fine-tune:")
acc, f1m = evaluate_model(cnn_ft)
print("Acc:", acc, "F1_macro:", f1m)

joblib.dump(le, os.path.join(OUT_DIR, "label_encoder_sender_labels.joblib"))


Class weights para sender_labels: [11.39272    0.5229511]
Epoch 1/8 - val_f1_macro: 0.3240
Epoch 2/8 - val_f1_macro: 0.5032
Epoch 3/8 - val_f1_macro: 0.5205
Epoch 4/8 - val_f1_macro: 0.4850
Epoch 5/8 - val_f1_macro: 0.4564
Epoch 6/8 - val_f1_macro: 0.5140
Epoch 7/8 - val_f1_macro: 0.5259
Epoch 8/8 - val_f1_macro: 0.5355
Epoch 1/8 - val_f1_macro: 0.4299
Epoch 2/8 - val_f1_macro: 0.4379
Epoch 3/8 - val_f1_macro: 0.4918
Epoch 4/8 - val_f1_macro: 0.5035
Epoch 5/8 - val_f1_macro: 0.4955
Epoch 6/8 - val_f1_macro: 0.5077
Epoch 7/8 - val_f1_macro: 0.5050
Epoch 8/8 - val_f1_macro: 0.4985
Evaluación CNN fine-tune:
              precision    recall  f1-score   support

           0       0.04      0.04      0.04       104
           1       0.96      0.96      0.96      2275

    accuracy                           0.92      2379
   macro avg       0.50      0.50      0.50      2379
weighted avg       0.92      0.92      0.92      2379

Acc: 0.9184531315678857 F1_macro: 0.4985113744034632


['diplomacy/models/deep\\label_encoder_sender_labels.joblib']

In [6]:
from imblearn.over_sampling import RandomOverSampler
import numpy as np

# --------------------------
# Oversampling de clase minoritaria
# --------------------------
ros = RandomOverSampler(random_state=42)

# Convertir a numpy arrays por si acaso
X_train_np = np.array(X_train)
y_train_np = np.array(y_train)

X_train_bal, y_train_bal = ros.fit_resample(X_train_np, y_train_np)

# Crear DataLoader con oversampling
train_ds_bal = DiplomacyDataset(X_train_bal, y_train_bal)
train_loader_bal = DataLoader(train_ds_bal, batch_size=BATCH_SIZE, shuffle=True)

print("Distribución después de oversampling:", np.bincount(y_train_bal))


Distribución después de oversampling: [9097 9097]


In [7]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from gensim.models import Word2Vec
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler
import joblib

# --------------------------
# Paths y Config
# --------------------------
DATA_PATH = "data/train_preprocessed.parquet"
EMB_DIR = "diplomacy/models/embeddings"
OUT_DIR = "diplomacy/models/deep"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 64
EPOCHS = 8
MAX_LEN = 120
EMB_DIM = 200  # debe coincidir con vector_size de Word2Vec

# --------------------------
# Cargar dataframe y etiquetas
# --------------------------
df = pd.read_parquet(DATA_PATH)

# Filtrar etiquetas válidas y convertir a binario
df = df[df["sender_labels"].isin(["true","false","True","False"])]
y_raw = df["sender_labels"].astype(str).str.lower().map({"true": 1, "false": 0})

le = LabelEncoder()
y = le.fit_transform(y_raw)

# --------------------------
# Pesos de clase
# --------------------------
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y),
    y=y
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print("Class weights para sender_labels:", class_weights.cpu().numpy())

# --------------------------
# Oversampling de la clase minoritaria
# --------------------------
ros = RandomOverSampler(random_state=42)
X_train_orig, X_val_orig, y_train_orig, y_val_orig = train_test_split(
    df["tokens"].values, y, test_size=0.2, random_state=42, stratify=y
)

# Convertir tokens a indices
w2v = Word2Vec.load(os.path.join(EMB_DIR, "word2vec.model"))

token2idx = {"<pad>":0, "<oov>":1}
for tok in w2v.wv.key_to_index.keys():
    token2idx[tok] = len(token2idx)
vocab_size = len(token2idx)

def to_indices(tokens):
    if isinstance(tokens, str):
        tokens = eval(tokens)
    idxs = [token2idx.get(tok, token2idx["<oov>"]) for tok in tokens]
    if len(idxs) >= MAX_LEN:
        return idxs[:MAX_LEN]
    else:
        return idxs + [token2idx["<pad>"]] * (MAX_LEN - len(idxs))

X_train_indices = np.vstack([np.array(to_indices(t)) for t in X_train_orig])
X_val_indices = np.vstack([np.array(to_indices(t)) for t in X_val_orig])

# Oversampling
X_train_bal, y_train_bal = ros.fit_resample(X_train_indices, y_train_orig)
print("Distribución después de oversampling:", np.bincount(y_train_bal))

# --------------------------
# Embedding matrix
# --------------------------
emb_matrix = np.random.normal(scale=0.6, size=(vocab_size, EMB_DIM)).astype(np.float32)
for tok, idx in token2idx.items():
    if tok in w2v.wv:
        emb_matrix[idx] = w2v.wv[tok]

# --------------------------
# Dataset y DataLoader
# --------------------------
class DiplomacyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds_bal = DiplomacyDataset(X_train_bal, y_train_bal)
val_ds = DiplomacyDataset(X_val_indices, y_val_orig)
train_loader_bal = DataLoader(train_ds_bal, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# --------------------------
# Modelo CNN
# --------------------------
class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, num_classes=2, freeze_emb=True, n_filters=100, kernel_sizes=[3,4,5]):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(emb_matrix))
        self.embedding.weight.requires_grad = not freeze_emb
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, n_filters, k) for k in kernel_sizes])
        self.fc = nn.Linear(n_filters * len(kernel_sizes), num_classes)
    def forward(self, x):
        e = self.embedding(x).permute(0,2,1)  # (B, emb_dim, L)
        convs = [torch.relu(conv(e)) for conv in self.convs]
        pools = [torch.max(c, dim=2)[0] for c in convs]
        cat = torch.cat(pools, dim=1)
        return self.fc(cat)

# --------------------------
# Entrenamiento
# --------------------------
def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_val_f1 = 0
    for epoch in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            opt.step()
        # Evaluación
        model.eval()
        y_trues, y_preds = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                logits = model(Xb)
                preds = logits.argmax(dim=1).cpu().numpy()
                y_preds.extend(preds)
                y_trues.extend(yb.cpu().numpy())
        val_f1 = f1_score(y_trues, y_preds, average="macro")
        print(f"Epoch {epoch+1}/{epochs} - val_f1_macro: {val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = model.state_dict()
    model.load_state_dict(best_state)
    return model

# --------------------------
# Entrenamiento final
# --------------------------
cnn_ft = CNNClassifier(vocab_size=vocab_size, emb_dim=EMB_DIM, num_classes=2, freeze_emb=False)
cnn_ft = train_model(cnn_ft, train_loader_bal, val_loader)
torch.save(cnn_ft.state_dict(), os.path.join(OUT_DIR, "cnn_sender_labels_balanced.pt"))

# --------------------------
# Evaluación
# --------------------------
def evaluate_model(model):
    model.eval()
    y_preds, y_trues = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            y_preds.extend(preds)
            y_trues.extend(yb.cpu().numpy())
    target_names = [str(c) for c in le.classes_]
    print(classification_report(y_trues, y_preds, target_names=target_names))
    return accuracy_score(y_trues, y_preds), f1_score(y_trues, y_preds, average="macro")

print("Evaluación CNN fine-tune:")
acc, f1m = evaluate_model(cnn_ft)
print("Acc:", acc, "F1_macro:", f1m)

joblib.dump(le, os.path.join(OUT_DIR, "label_encoder_sender_labels.joblib"))


Class weights para sender_labels: [11.39272    0.5229511]
Distribución después de oversampling: [9097 9097]
Epoch 1/8 - val_f1_macro: 0.3473
Epoch 2/8 - val_f1_macro: 0.5071
Epoch 3/8 - val_f1_macro: 0.5017
Epoch 4/8 - val_f1_macro: 0.5150
Epoch 5/8 - val_f1_macro: 0.5165
Epoch 6/8 - val_f1_macro: 0.5148
Epoch 7/8 - val_f1_macro: 0.5074
Epoch 8/8 - val_f1_macro: 0.5079
Evaluación CNN fine-tune:
              precision    recall  f1-score   support

           0       0.06      0.10      0.07       104
           1       0.96      0.93      0.94      2275

    accuracy                           0.89      2379
   macro avg       0.51      0.51      0.51      2379
weighted avg       0.92      0.89      0.91      2379

Acc: 0.89281210592686 F1_macro: 0.507922860097743


['diplomacy/models/deep\\label_encoder_sender_labels.joblib']

### **2. Predicción del hablante**


In [9]:
"""
Predicción del hablante con modelos deep:
- LSTM (embedding inicializado con Word2Vec; opción freeze=True/False)
- CNN 1D (idem)
Guarda modelos y métricas.
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from gensim.models import Word2Vec
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
import joblib
from tqdm import tqdm

# Paths
EMB_DIR = "diplomacy/models/embeddings"
OUT_DIR = "diplomacy/models/deep"
os.makedirs(OUT_DIR, exist_ok=True)

# Config
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 64
EPOCHS = 8
MAX_LEN = 120  # to pad/truncate sequences
EMB_DIM = 200  # must match your Word2Vec vector_size

# Cargar dataframe
df = pd.read_parquet("data/train_preprocessed.parquet")

# Detectar columna de hablante
if "speaker" in df.columns:
    speaker_col = "speaker"
elif "speakers" in df.columns:
    speaker_col = "speakers"
else:
    raise RuntimeError("No se encuentra columna 'speaker' ni 'speakers'.")

# ============================
# 🔥 Filtrar clases raras (mínimo 2 ejemplos)
# ============================
counts = df[speaker_col].value_counts()
valid_speakers = counts[counts >= 2].index

df = df[df[speaker_col].isin(valid_speakers)].reset_index(drop=True)

print("Speakers válidos:", len(valid_speakers))
print(counts[counts < 2], "\nTodos los anteriores fueron eliminados.")

# Label encoding
le = LabelEncoder()
y = le.fit_transform(df[speaker_col].astype(str))

# Load Word2Vec (pretrained on your corpus)
w2v = Word2Vec.load(os.path.join(EMB_DIR, "word2vec.model"))

# Build vocabulary -> token to index (reserve 0 for padding, 1 for OOV)
token2idx = {"<pad>":0, "<oov>":1}
for token in w2v.wv.key_to_index.keys():
    token2idx[token] = len(token2idx)

vocab_size = len(token2idx)
print("Vocab size (embedding):", vocab_size)

# Token sequences: df['tokens'] should be list or string of list
def to_indices(tokens):
    if isinstance(tokens, str):
        tokens = eval(tokens)
    idxs = [token2idx.get(tok, token2idx["<oov>"]) for tok in tokens]
    # pad/truncate
    if len(idxs) >= MAX_LEN:
        return idxs[:MAX_LEN]
    else:
        return idxs + [token2idx["<pad>"]] * (MAX_LEN - len(idxs))

X_seq = np.vstack(df["tokens"].apply(lambda t: np.array(to_indices(t))).values)
# split
X_train, X_val, y_train, y_val = train_test_split(X_seq, y, test_size=0.2, random_state=42, stratify=y)

# Create embedding matrix from Word2Vec
emb_matrix = np.random.normal(scale=0.6, size=(vocab_size, EMB_DIM)).astype(np.float32)
for tok, idx in token2idx.items():
    if tok in w2v.wv:
        emb_matrix[idx] = w2v.wv[tok]

# Dataset
class SpeakerDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = SpeakerDataset(X_train, y_train)
val_ds = SpeakerDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# Models
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden=128, n_layers=1, freeze_emb=True, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(emb_matrix))
        self.embedding.weight.requires_grad = not freeze_emb
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, num_classes)
    def forward(self, x):
        e = self.embedding(x)
        out, _ = self.lstm(e)
        # mean pool
        out = out.mean(dim=1)
        return self.fc(out)

class CNNClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, num_classes=2, freeze_emb=True, n_filters=100, kernel_sizes=[3,4,5]):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.embedding.weight.data.copy_(torch.from_numpy(emb_matrix))
        self.embedding.weight.requires_grad = not freeze_emb
        self.convs = nn.ModuleList([nn.Conv1d(emb_dim, n_filters, k) for k in kernel_sizes])
        self.fc = nn.Linear(n_filters * len(kernel_sizes), num_classes)
    def forward(self, x):
        e = self.embedding(x).permute(0,2,1)  # (B, emb_dim, L)
        convs = [torch.relu(conv(e)) for conv in self.convs]
        pools = [torch.max(c, dim=2)[0] for c in convs]
        cat = torch.cat(pools, dim=1)
        return self.fc(cat)

# training helper
def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    best_val_f1 = 0
    for epoch in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logits = model(Xb)
            loss = criterion(logits, yb)
            loss.backward()
            opt.step()
        # eval
        model.eval()
        y_trues, y_preds = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb = Xb.to(DEVICE)
                logits = model(Xb)
                preds = logits.argmax(dim=1).cpu().numpy()
                y_preds.extend(preds)
                y_trues.extend(yb.numpy())
        val_f1 = f1_score(y_trues, y_preds, average="macro")
        print(f"Epoch {epoch+1}/{epochs} - val_f1_macro: {val_f1:.4f}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = model.state_dict()
    model.load_state_dict(best_state)
    return model

# number of classes
num_classes = len(le.classes_)

# Train LSTM (freeze embeddings first)
print("Entrenando LSTM (embeddings congelados)...")
lstm = LSTMClassifier(vocab_size=vocab_size, emb_dim=EMB_DIM, num_classes=num_classes, freeze_emb=True)
lstm = train_model(lstm, train_loader, val_loader)
torch.save(lstm.state_dict(), os.path.join(OUT_DIR, "lstm_freeze.pt"))

# Train LSTM (fine-tune)
print("Entrenando LSTM (embeddings fine-tune)...")
lstm_ft = LSTMClassifier(vocab_size=vocab_size, emb_dim=EMB_DIM, num_classes=num_classes, freeze_emb=False)
lstm_ft = train_model(lstm_ft, train_loader, val_loader)
torch.save(lstm_ft.state_dict(), os.path.join(OUT_DIR, "lstm_finetune.pt"))

# Train CNN (freeze)
print("Entrenando CNN (embeddings congelados)...")
cnn = CNNClassifier(vocab_size=vocab_size, emb_dim=EMB_DIM, num_classes=num_classes, freeze_emb=True)
cnn = train_model(cnn, train_loader, val_loader)
torch.save(cnn.state_dict(), os.path.join(OUT_DIR, "cnn_freeze.pt"))

# Train CNN (fine-tune)
print("Entrenando CNN (embeddings fine-tune)...")
cnn_ft = CNNClassifier(vocab_size=vocab_size, emb_dim=EMB_DIM, num_classes=num_classes, freeze_emb=False)
cnn_ft = train_model(cnn_ft, train_loader, val_loader)
torch.save(cnn_ft.state_dict(), os.path.join(OUT_DIR, "cnn_finetune.pt"))

# Evaluate final model examples (LSTM fine-tune)
def evaluate_model(model):
    model.eval()
    y_preds = []
    y_trues = []

    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            y_preds.extend(preds)
            y_trues.extend(yb.numpy())

    y_preds = np.array(y_preds)
    y_trues = np.array(y_trues)

    # ==============================
    # 🔥 Filtrar solo clases presentes
    # ==============================
    labels_present = np.unique(y_trues)
    target_names_present = [le.classes_[i] for i in labels_present]

    print(classification_report(
        y_trues,
        y_preds,
        labels=labels_present,
        target_names=target_names_present
    ))

    return accuracy_score(y_trues, y_preds), f1_score(y_trues, y_preds, average="macro")

print("Evaluación LSTM fine-tune:")
acc, f1m = evaluate_model(lstm_ft)
print("Acc:", acc, "F1_macro:", f1m)
joblib.dump(le, os.path.join(OUT_DIR, "label_encoder.joblib"))

Speakers válidos: 179
speakers
['russia', 'france']    1
['austria']             1
['russia']              1
['germany']             1
Name: count, dtype: int64 
Todos los anteriores fueron eliminados.
Vocab size (embedding): 2027
Entrenando LSTM (embeddings congelados)...
Epoch 1/8 - val_f1_macro: 0.0016
Epoch 2/8 - val_f1_macro: 0.0015
Epoch 3/8 - val_f1_macro: 0.0022
Epoch 4/8 - val_f1_macro: 0.0041
Epoch 5/8 - val_f1_macro: 0.0052
Epoch 6/8 - val_f1_macro: 0.0060
Epoch 7/8 - val_f1_macro: 0.0074
Epoch 8/8 - val_f1_macro: 0.0088
Entrenando LSTM (embeddings fine-tune)...
Epoch 1/8 - val_f1_macro: 0.0011
Epoch 2/8 - val_f1_macro: 0.0017
Epoch 3/8 - val_f1_macro: 0.0045
Epoch 4/8 - val_f1_macro: 0.0053
Epoch 5/8 - val_f1_macro: 0.0092
Epoch 6/8 - val_f1_macro: 0.0111
Epoch 7/8 - val_f1_macro: 0.0140
Epoch 8/8 - val_f1_macro: 0.0174
Entrenando CNN (embeddings congelados)...
Epoch 1/8 - val_f1_macro: 0.0047
Epoch 2/8 - val_f1_macro: 0.0108
Epoch 3/8 - val_f1_macro: 0.0111
Epoch 4/8 - val

c:\Users\Oihane\Desktop\NLP\Entrega2\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Oihane\Desktop\NLP\Entrega2\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Oihane\Desktop\NLP\Entrega2\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

['diplomacy/models/deep\\label_encoder.joblib']